In [ ]:
from IPython.display import Markdown, display


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from src.ingestion.pdf_parser import parse_pdf
from src.ingestion.chunker import load_and_chunk_pdf
from src.ingestion.embedder import embed_texts
from src.ingestion.ingest_pipeline import run_ingestion
from src.retrieval.retriever import retrieve
from src.retrieval.reranker import rerank
from src.retrieval.vector_store import VectorStore

To resolve the file path and load the pdf using env

In [ ]:
load_dotenv()
ROOT = Path(os.getenv("PYTHONPATH"))
# Resolve path from project root (notebook often runs with cwd = notebooks/)
ROOT = Path.cwd()
if not (ROOT / "uploaded_pdfs").exists():
    ROOT = ROOT.parent

To connecting the VectorDB

In [ ]:
vs = VectorStore()
print("ChromaDB connected")

Adding all the papers in the uploaded_pdfs. 

 **Dont run this multiple times it might cost you more money(Obviously TOKENS) because each time you add a duplicate it will delete the old one and replaces it with this one**

For Deleting all papers in CHROMA DB, **dont run this unless you need it**

In [ ]:
import shutil
from pathlib import Path

shutil.rmtree(ROOT / "chroma_db")
print("ChromaDB wiped")

To load all the datasets in to the DB.

In [ ]:
pdf_folder = ROOT / "uploaded_pdfs"

for pdf in pdf_folder.glob("*.pdf"):
    print(f"Ingesting: {pdf.name}")
    result = run_ingestion(str(pdf), source="system")
    print(f"Done — {result['num_chunks']} chunks")

To ingest a single new system paper without re-running the whole folder.

In [ ]:
# Change this to the filename of the new paper inside uploaded_pdfs/
NEW_PAPER = "your_paper.pdf"

pdf_path = ROOT / "uploaded_pdfs" / NEW_PAPER
result = run_ingestion(str(pdf_path), source="system")
print(f"Done — {result['num_chunks']} chunks ingested for '{pdf_path.stem}'")

Test

In [ ]:
pdf_path = ROOT / "uploaded_pdfs" / "Attention_is _all_you_need.pdf"

# parse and chunk
chunks = load_and_chunk_pdf(str(pdf_path))
print(f"Total chunks: {len(chunks)}")

# embed
vectors = embed_texts(chunks)
print(f"Total vectors: {len(vectors)}")

# save to ChromaDB
vs.add(chunks, vectors, "attention")
print("Saved to ChromaDB")

This **.add** will add the pdf in VS by passsing the hash function i wrote later on in this project so we still can have duplicates if we add manually take care of that issue if you notice.

In [ ]:
#vs.add(chunks, vectors, "attention")
#print("Saved to ChromaDB")

Testing the retriever.py

In [ ]:
query = "What is the attention mechanism?"
query_vector = embed_texts([query])[0]

results = vs.search(query_vector, top_k=5)

for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result["text"][:200])
    print(f"Score: {result['score']}")
    print("---")

We got 5 papers because the papers are distinguished with the paper name (Need to chnage this later in the project to remove duplicates automatically.)

I changed it and reduced the duplicates using hash.

**Just removed the cells where i removed those duplicates**

In [ ]:
from src.retrieval.vector_store import VectorStore
vs = VectorStore()
vs.list_papers()  # see the actual names

Testing wether the duplicate detcion is working or not.

In [ ]:
result = run_ingestion("uploaded_pdfs/Attention_is _all_you_need.pdf")
print(result)

The Duplicates removal has worked for now 

In [ ]:
results = retrieve("What is the attention mechanism?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("What is this project?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("What is paris located in the world?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("Does agentic retrieval works?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("what are the uses of RAG")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("what is RAG")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("How does retrieval-augmented generation combine retrieval with generation")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

After implementing the RERANKER

In [ ]:
question = "How does retrieval-augmented generation combine retrieval with generation"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)

for i, chunk in enumerate(reranked):
    print(f"Result {i+1} (score: {chunk['score']:.4f}):")
    display(Markdown(chunk["text"]))
    print("---")

In [ ]:
question = "What is the attention mechanism?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)

for i, chunk in enumerate(reranked):
    print(f"Result {i+1} (score: {chunk['score']:.4f}):")
    display(Markdown(chunk["text"]))
    print("---")

Just to check the prompt_builder.py file working or not

In [ ]:
from src.generation.prompt_builder import build_prompt

question = "What is the attention mechanism?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)

prompt = build_prompt(question, reranked)
print(prompt)

To check wether Llm_router.py is working.


In [ ]:
from src.generation.llm_router import generate

answer = generate("What is the attention mechanism?")
display(Markdown(answer))

Dont get shocked haha, Llm_router.py only know to call the llm directly which doesnt get any information from RAG yet so its saying it doesnt know.

In [ ]:
from src.retrieval.retriever import retrieve
from src.retrieval.reranker import rerank
from src.generation.prompt_builder import build_prompt
from src.generation.llm_router import generate

question = "What is the attention mechanism?"

# 1. retrieve
results = retrieve(question)

# 2. rerank
reranked = rerank(question, results, top_n=5)

# 3. build prompt with chunks
prompt = build_prompt(question, reranked)

# 4. generate
answer = generate(prompt)
display(Markdown(answer))

Finnally testing the generator.py wrapping

In [ ]:
from src.retrieval.retriever import retrieve
from src.retrieval.reranker import rerank
from src.generation.generator import generate_answer



In [ ]:
question = "Who is Murali krishna and does he know RAG(retrieval argumentative generator ?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked)


display(Markdown(answer["answer"]))
print(f"\nModel: {answer['model']}")
print(f"Tokens: {answer['token_count']}")
print(f"Chunks used: {len(answer['source_chunks'])}")

In [ ]:
question = "Are you able to research and help me build a RAG?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked)


display(Markdown(answer["answer"]))
print(f"\nModel: {answer['model']}")
print(f"Tokens: {answer['token_count']}")
print(f"Chunks used: {len(answer['source_chunks'])}")

In [ ]:
question = "What is the attention mechanism?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked)


display(Markdown(answer["answer"]))
print(f"\nModel: {answer['model']}")
print(f"Tokens: {answer['token_count']}")
print(f"Chunks used: {len(answer['source_chunks'])}")

In [ ]:
question = "What is the attention mechanism?"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked, model="claude-haiku-4-5-20251001")

display(Markdown(answer["answer"]))
print(f"\nModel: {answer['model']}")
print(f"Tokens: {answer['token_count']}")
print(f"Chunks used: {len(answer['source_chunks'])}")

In [ ]:
question = "What is Murali krishna and does he know RAG(retrieval argumentative generator ??"
results = retrieve(question)
reranked = rerank(question, results, top_n=5)
answer = generate_answer(question, reranked, model="claude-haiku-4-5-20251001")

display(Markdown(answer["answer"]))
print(f"\nModel: {answer['model']}")
print(f"Tokens: {answer['token_count']}")
print(f"Chunks used: {len(answer['source_chunks'])}")

to test eval_dataest.py

In [8]:
from src.evaluation.eval_dataset import load_eval_dataset

ds = load_eval_dataset("../eval_data/test_questions.json")
print(f"Total questions: {len(ds)}")
print(ds[0])

Total questions: 23
{'question': 'What BLEU score did the Transformer achieve on WMT 2014 English-to-German?', 'ground_truth': 'The Transformer achieved 28.4 BLEU on the WMT 2014 English-to-German translation task.', 'answer': '', 'contexts': []}
